# 과제 3 — 자연어 → 모듈 조합

결정론적 게이트 루프를 포함한 풀이 노트북입니다. 일부 후보식은 실패 과정을 보여주기 위해 의도적으로 잘못 작성했습니다.

### 1. 접근 방식

이 과제는 자연어 규칙을 자유로운 쿼리 문법으로 생성하는 문제가 아니라, 제공된 안전한 함수만 조합하는 문제로 해석했다.   
따라서 표현식은 문자열 파싱 대신 Python tuple 기반 AST로 작성했다.

예시는 다음과 같다.

```python
("forEvery", ("allOf", "Pump"), ("isAtLeast", ("prop", "x", "DesignPressure"), 5))
```

이 방식의 장점은 다음과 같다.

- 함수명, 인자 개수, 인자 타입을 결정론적으로 검사할 수 있다.
- LLM이 만든 자유 문법을 바로 실행하지 않고, 게이트가 통과한 AST만 후속 실행기로 넘길 수 있다.
- 실패 사유를 구체적으로 피드백할 수 있어 재생성 루프를 구성하기 쉽다.

### 2. 가정

| 항목 | 가정 |
|---|---|
| vessel | `Vessel` 클래스로 매핑한다. Tank까지 포함하는지는 실제 온톨로지 정의에 따라 달라질 수 있다. |
| 모든 장비 | `Equipment` 클래스로 매핑한다. |
| 열교환기 | `HeatExchanger` 클래스로 매핑한다. |
| 압축기 | `Compressor` 클래스로 매핑한다. |
| 설계압력 | `DesignPressure` |
| 운전압력 | `OperatingPressure` |
| 설계온도 | `DesignTemperature` |
| 운전온도 | `OperatingTemperature` |
| 재질 | `Material` |
| 태그 | `Tag` |
| 태그 패턴 | `TAG_PATTERN` |
| 정의되어 있음 | `exists(prop(x, ...))` |
| A보다 30 이상 높다 | `isAtLeast(A, plus(B, 30))` |
| 조건부 규칙 | 별도 `where`/`implies`가 없으므로 `or(not(조건), 결론)`으로 표현한다. |

### 3. 게이트 설계

검증기는 다음을 검사한다.

1. 최상위 함수가 `forEvery`인지 검사한다.
2. 명세에 없는 함수명이 있는지 검사한다.
3. 각 함수의 인자 개수를 검사한다.
4. `forEvery`의 첫 번째 인자는 selector인지 검사한다.
5. selector는 `allOf(className)` 또는 `withSafetyValve(className)`만 허용한다.
6. condition은 비교, 존재성, 패턴 매칭, 논리 결합만 허용한다.
7. 비교 연산의 피연산자는 `prop(...)`, 리터럴, `times(...)`, `plus(...)`로 귀결되어야 한다.
8. className, propertyName, regexName은 허용 목록에 있어야 한다.

In [1]:
import pandas as pd
from dataclasses import dataclass
from typing import Any, List, Tuple, Union


Expr = Union[tuple, str, int, float]

ALLOWED_FUNCS = {"allOf", "withSafetyValve", "prop", "isAtLeast", "isAtMost", "isEqual", "exists", 
                 "matchesPattern", "times", "plus", "and", "or", "not", "forEvery"}
CLASS_NAMES = {"Equipment", "Vessel", "Tank", "Pump", "HeatExchanger", "Compressor"}
PROPERTY_NAMES = {"DesignPressure", "OperatingPressure", "DesignTemperature", "OperatingTemperature", "Tag", "Material"}
REGEX_NAMES = {"TAG_PATTERN"}


@dataclass
class ValidationResult:
    ok: bool
    errors: List[str]


def is_literal(x:Any) -> bool:
    return isinstance(x, (int, float, str)) and x != "x"


def func_name(expr:Any) -> str:
    if isinstance(expr, tuple) and len(expr) > 0:
        return expr[0]
    return ""


def validate_expr(expr:Expr, expected:str, path:str="root") -> List[str]:
    errors = []

    if expected == "value":
        if is_literal(expr):
            return []
        if not isinstance(expr, tuple):
            return [f"{path}: value가 필요하지만 {expr!r} 입니다."]
        f = func_name(expr)
        if f == "prop":
            if len(expr) != 3:
                return [f"{path}: prop는 인자 2개가 필요합니다. 현재 {len(expr)-1}개"]
            _, target, prop_name = expr
            if target != "x":
                errors.append(f"{path}: prop의 첫 번째 인자는 반복 변수 x여야 합니다.")
            if prop_name not in PROPERTY_NAMES:
                errors.append(f"{path}: 허용되지 않은 속성명 {prop_name!r}")
            return errors
        if f in {"times", "plus"}:
            if len(expr) != 3:
                return [f"{path}: {f}는 인자 2개가 필요합니다. 현재 {len(expr)-1}개"]
            _, a, k = expr
            errors += validate_expr(a, "value", path + f".{f}[0]")
            if not isinstance(k, (int, float)):
                errors.append(f"{path}: {f}의 두 번째 인자는 숫자 리터럴이어야 합니다.")
            return errors
        return [f"{path}: value 위치에 허용되지 않은 함수 {f!r}가 사용되었습니다."]

    if expected == "selector":
        if not isinstance(expr, tuple):
            return [f"{path}: selector는 함수 호출이어야 합니다."]
        f = func_name(expr)
        if f not in {"allOf", "withSafetyValve"}:
            return [f"{path}: selector 위치에는 allOf/withSafetyValve만 올 수 있습니다. 현재 {f!r}"]
        if len(expr) != 2:
            return [f"{path}: {f}는 className 인자 1개가 필요합니다. 현재 {len(expr)-1}개"]
        class_name = expr[1]
        if class_name not in CLASS_NAMES:
            errors.append(f"{path}: 허용되지 않은 className {class_name!r}")
        return errors

    if expected == "condition":
        if not isinstance(expr, tuple):
            return [f"{path}: condition은 함수 호출이어야 합니다."]
        f = func_name(expr)
        if f not in ALLOWED_FUNCS:
            return [f"{path}: 명세에 없는 함수 {f!r}가 사용되었습니다."]
        if f in {"isAtLeast", "isAtMost", "isEqual"}:
            if len(expr) != 3:
                return [f"{path}: {f}는 인자 2개가 필요합니다. 현재 {len(expr)-1}개"]
            errors += validate_expr(expr[1], "value", path + f".{f}[0]")
            errors += validate_expr(expr[2], "value", path + f".{f}[1]")
            return errors
        if f == "exists":
            if len(expr) != 2:
                return [f"{path}: exists는 인자 1개가 필요합니다. 현재 {len(expr)-1}개"]
            return validate_expr(expr[1], "value", path + ".exists[0]")
        if f == "matchesPattern":
            if len(expr) != 3:
                return [f"{path}: matchesPattern은 인자 2개가 필요합니다. 현재 {len(expr)-1}개"]
            errors += validate_expr(expr[1], "value", path + ".matchesPattern[0]")
            if expr[2] not in REGEX_NAMES:
                errors.append(f"{path}: 허용되지 않은 regexName {expr[2]!r}")
            return errors
        if f in {"and", "or"}:
            if len(expr) < 3:
                return [f"{path}: {f}는 최소 2개의 조건 인자가 필요합니다."]
            for i, child in enumerate(expr[1:]):
                errors += validate_expr(child, "condition", path + f".{f}[{i}]")
            return errors
        if f == "not":
            if len(expr) != 2:
                return [f"{path}: not은 인자 1개가 필요합니다. 현재 {len(expr)-1}개"]
            return validate_expr(expr[1], "condition", path + ".not[0]")
        return [f"{path}: condition 위치에 허용되지 않은 함수 {f!r}가 사용되었습니다."]

    return [f"{path}: 알 수 없는 expected type {expected!r}"]


def validate_rule(expr:Expr) -> ValidationResult:
    errors = []
    if not isinstance(expr, tuple) or len(expr) == 0:
        return ValidationResult(False, ["최상위 표현식은 forEvery(...) 함수 호출이어야 합니다."])
    if func_name(expr) != "forEvery":
        errors.append(f"최상위 함수는 forEvery여야 합니다. 현재 {func_name(expr)!r}")
    if func_name(expr) not in ALLOWED_FUNCS:
        errors.append(f"명세에 없는 최상위 함수 {func_name(expr)!r}가 사용되었습니다.")
    if len(expr) != 3:
        errors.append(f"forEvery는 target, condition 인자 2개가 필요합니다. 현재 {len(expr)-1}개")
        return ValidationResult(False, errors)
    errors += validate_expr(expr[1], "selector", "root.forEvery.target")
    errors += validate_expr(expr[2], "condition", "root.forEvery.condition")
    return ValidationResult(len(errors) == 0, errors)


def s(expr:Expr) -> str:
    if isinstance(expr, tuple):
        return expr[0] + "(" + ", ".join(s(x) for x in expr[1:]) + ")"
    if isinstance(expr, str):
        return f'"{expr}"' if expr != "x" else "x"
    return str(expr)

### 4. 실패/재생성 로그 요약

각 문장마다 일부러 실패 후보를 1개 이상 넣었다. 게이트가 실패 사유를 반환하면 다음 후보를 검사하고, 통과한 식을 최종식으로 채택했다.

| 번호 | 실패 예시 | 실패 이유 | 재생성 후 처리 |
|---:|---|---|---|
| 1 | `all("Vessel")` | selector 함수가 명세에 없음 | `allOf("Vessel")`로 수정 |
| 2 | `defined(...)` | 명세에 없는 함수 | `exists(...)`로 수정 |
| 3 | `greaterThan(...)` | 명세에 없는 비교 함수 | `isAtLeast(..., plus(..., 30))`로 수정 |
| 4 | `matches(...)` | 명세에 없는 함수 | `matchesPattern(...)`으로 수정 |
| 5 | `and(...)`에 조건 1개만 전달 | `and`는 최소 2개 조건 필요 | 재질/설계압력 exists 2개로 수정 |
| 6 | 최상위가 `not(...)` | 최상위는 `forEvery`여야 함 | `forEvery(allOf("Tank"), isAtMost(..., 2))`로 수정 |
| 7 | `where(...)` | selector 위치에 허용되지 않은 함수 | `or(not(조건), 결론)`으로 조건부 규칙 표현 |
| 8 | `multiply(...)` | 산술 함수명이 명세와 다름 | `times(...)`로 수정 |
| 9 | `forEvery` 인자 3개 | 조건 2개를 `and`로 묶지 않음 | `and(조건1, 조건2)`로 수정 |
| 10 | `in(...)` | 명세에 없는 membership 함수 | `or(isEqual(...), isEqual(...))`로 수정 |
| 11 | `isLess(...)` | 명세에 없는 비교 함수 | `isAtLeast(DesignTemperature, OperatingTemperature)`로 수정 |
| 12 | `TAG_REGEX` | 허용되지 않은 regexName | `TAG_PATTERN`으로 수정 |

모든 문장은 실패 후보 1회 후 최종 후보가 통과하도록 구성했다. 따라서 전체 재생성 횟수는 12회, 최종 통과율은 12/12 = 100%다.

##### 4-1. 후보식 생성 

In [2]:
CANDIDATES = [
    {
        "id": 1,
        "sentence": "모든 용기(vessel)의 설계압력은 운전압력의 1.1배 이상이어야 한다.",
        "candidates": [
            ("v1_fail", ("forEvery", ("all", "Vessel"), ("isAtLeast", ("prop", "x", "DesignPressure"), ("times", ("prop", "x", "OperatingPressure"), 1.1)))),
            ("v2_pass", ("forEvery", ("allOf", "Vessel"), ("isAtLeast", ("prop", "x", "DesignPressure"), ("times", ("prop", "x", "OperatingPressure"), 1.1))))
        ]
    },
    {
        "id": 2,
        "sentence": "안전밸브가 있는 모든 장비는 설계압력이 정의되어 있어야 한다.",
        "candidates": [
            ("v1_fail", ("forEvery", ("withSafetyValve", "Equipment"), ("defined", ("prop", "x", "DesignPressure")))),
            ("v2_pass", ("forEvery", ("withSafetyValve", "Equipment"), ("exists", ("prop", "x", "DesignPressure"))))
        ]
    },
    {
        "id": 3,
        "sentence": "모든 열교환기의 설계온도는 운전온도보다 30 이상 높아야 한다.",
        "candidates": [
            ("v1_fail", ("forEvery", ("allOf", "HeatExchanger"), ("greaterThan", ("prop", "x", "DesignTemperature"), ("plus", ("prop", "x", "OperatingTemperature"), 30)))),
            ("v2_pass", ("forEvery", ("allOf", "HeatExchanger"), ("isAtLeast", ("prop", "x", "DesignTemperature"), ("plus", ("prop", "x", "OperatingTemperature"), 30))))
        ]
    },
    {
        "id": 4,
        "sentence": "모든 장비 태그는 정해진 태그 패턴과 일치해야 한다.",
        "candidates": [
            ("v1_fail", ("forEvery", ("allOf", "Equipment"), ("matches", ("prop", "x", "Tag"), "TAG_PATTERN"))),
            ("v2_pass", ("forEvery", ("allOf", "Equipment"), ("matchesPattern", ("prop", "x", "Tag"), "TAG_PATTERN")))
        ]
    },
    {
        "id": 5,
        "sentence": "모든 펌프는 재질과 설계압력이 모두 정의되어 있어야 한다.",
        "candidates": [
            ("v1_fail", ("forEvery", ("allOf", "Pump"), ("and", ("exists", ("prop", "x", "Material"))))),
            ("v2_pass", ("forEvery", ("allOf", "Pump"), ("and", ("exists", ("prop", "x", "Material")), ("exists", ("prop", "x", "DesignPressure")))))
        ]
    },
    {
        "id": 6,
        "sentence": "어떤 탱크도 운전압력이 2 barg를 초과해서는 안 된다.",
        "candidates": [
            ("v1_fail", ("not", ("exists", ("isGreater", ("prop", "x", "OperatingPressure"), 2)))),
            ("v2_pass", ("forEvery", ("allOf", "Tank"), ("isAtMost", ("prop", "x", "OperatingPressure"), 2)))
        ]
    },
    {
        "id": 7,
        "sentence": "운전온도가 200 이상인 모든 용기는 설계온도가 240 이상이어야 한다.",
        "candidates": [
            ("v1_fail", ("forEvery", ("where", ("allOf", "Vessel"), ("isAtLeast", ("prop", "x", "OperatingTemperature"), 200)), ("isAtLeast", ("prop", "x", "DesignTemperature"), 240))),
            ("v2_pass", ("forEvery", ("allOf", "Vessel"), ("or", ("not", ("isAtLeast", ("prop", "x", "OperatingTemperature"), 200)), ("isAtLeast", ("prop", "x", "DesignTemperature"), 240))))
        ]
    },
    {
        "id": 8,
        "sentence": "안전밸브가 있는 모든 장비는 설계압력이 운전압력의 1.2배 이상이어야 한다.",
        "candidates": [
            ("v1_fail", ("forEvery", ("withSafetyValve", "Equipment"), ("isAtLeast", ("prop", "x", "DesignPressure"), ("multiply", ("prop", "x", "OperatingPressure"), 1.2)))),
            ("v2_pass", ("forEvery", ("withSafetyValve", "Equipment"), ("isAtLeast", ("prop", "x", "DesignPressure"), ("times", ("prop", "x", "OperatingPressure"), 1.2))))
        ]
    },
    {
        "id": 9,
        "sentence": "모든 압축기는 설계압력이 운전압력의 1.1배 이상이고, 동시에 설계온도가 운전온도보다 30 이상 높아야 한다.",
        "candidates": [
            ("v1_fail", ("forEvery", ("allOf", "Compressor"), ("isAtLeast", ("prop", "x", "DesignPressure"), ("times", ("prop", "x", "OperatingPressure"), 1.1)), ("isAtLeast", ("prop", "x", "DesignTemperature"), ("plus", ("prop", "x", "OperatingTemperature"), 30)))),
            ("v2_pass", ("forEvery", ("allOf", "Compressor"), ("and", ("isAtLeast", ("prop", "x", "DesignPressure"), ("times", ("prop", "x", "OperatingPressure"), 1.1)), ("isAtLeast", ("prop", "x", "DesignTemperature"), ("plus", ("prop", "x", "OperatingTemperature"), 30)))))
        ]
    },
    {
        "id": 10,
        "sentence": "모든 열교환기의 재질은 SS304 또는 SS316 중 하나여야 한다.",
        "candidates": [
            ("v1_fail", ("forEvery", ("allOf", "HeatExchanger"), ("in", ("prop", "x", "Material"), ["SS304", "SS316"]))),
            ("v2_pass", ("forEvery", ("allOf", "HeatExchanger"), ("or", ("isEqual", ("prop", "x", "Material"), "SS304"), ("isEqual", ("prop", "x", "Material"), "SS316"))))
        ]
    },
    {
        "id": 11,
        "sentence": "어떤 장비도 설계온도가 운전온도보다 낮아서는 안 된다.",
        "candidates": [
            ("v1_fail", ("forEvery", ("allOf", "Equipment"), ("not", ("isLess", ("prop", "x", "DesignTemperature"), ("prop", "x", "OperatingTemperature"))))),
            ("v2_pass", ("forEvery", ("allOf", "Equipment"), ("isAtLeast", ("prop", "x", "DesignTemperature"), ("prop", "x", "OperatingTemperature"))))
        ]
    },
    {
        "id": 12,
        "sentence": "모든 펌프 태그는 태그 패턴과 일치해야 하고, 설계압력이 정의되어 있어야 한다.",
        "candidates": [
            ("v1_fail", ("forEvery", ("allOf", "Pump"), ("and", ("matchesPattern", ("prop", "x", "Tag"), "TAG_REGEX"), ("exists", ("prop", "x", "DesignPressure"))))),
            ("v2_pass", ("forEvery", ("allOf", "Pump"), ("and", ("matchesPattern", ("prop", "x", "Tag"), "TAG_PATTERN"), ("exists", ("prop", "x", "DesignPressure")))))
        ]
    },
]

##### 4-2. 게이트 루프 실행

In [ ]:
run_log = []
final_rules = []

for item in CANDIDATES:
    accepted = None
    failures = []
    for version, expr in item["candidates"]:
        result = validate_rule(expr)
        if result.ok:
            accepted = (version, expr)
            break
        failures.append({"version":version, "expr":s(expr), "errors":result.errors})
    final_rules.append({
        "id": item["id"],
        "sentence": item["sentence"],
        "accepted_version": accepted[0] if accepted else None,
        "final_expr": s(accepted[1]) if accepted else None,
        "regen_count": len(failures),
        "failures": failures,
    })

for r in final_rules:
    print(f"[{r['id']}] 재생성 {r['regen_count']}회 → {r['accepted_version']}")
    print(r['final_expr'])

[1] 재생성 1회 → v2_pass
forEvery(allOf("Vessel"), isAtLeast(prop(x, "DesignPressure"), times(prop(x, "OperatingPressure"), 1.1)))
[2] 재생성 1회 → v2_pass
forEvery(withSafetyValve("Equipment"), exists(prop(x, "DesignPressure")))
[3] 재생성 1회 → v2_pass
forEvery(allOf("HeatExchanger"), isAtLeast(prop(x, "DesignTemperature"), plus(prop(x, "OperatingTemperature"), 30)))
[4] 재생성 1회 → v2_pass
forEvery(allOf("Equipment"), matchesPattern(prop(x, "Tag"), "TAG_PATTERN"))
[5] 재생성 1회 → v2_pass
forEvery(allOf("Pump"), and(exists(prop(x, "Material")), exists(prop(x, "DesignPressure"))))
[6] 재생성 1회 → v2_pass
forEvery(allOf("Tank"), isAtMost(prop(x, "OperatingPressure"), 2))
[7] 재생성 1회 → v2_pass
forEvery(allOf("Vessel"), or(not(isAtLeast(prop(x, "OperatingTemperature"), 200)), isAtLeast(prop(x, "DesignTemperature"), 240)))
[8] 재생성 1회 → v2_pass
forEvery(withSafetyValve("Equipment"), isAtLeast(prop(x, "DesignPressure"), times(prop(x, "OperatingPressure"), 1.2)))
[9] 재생성 1회 → v2_pass
forEvery(allOf("Compressor"),

##### 4-3. 최종 통과식 표

In [4]:
rows=[]

for r in final_rules:
    rows.append({
        "번호": r["id"],
        "문장": r["sentence"],
        "재생성횟수": r["regen_count"],
        "최종식": r["final_expr"],
        "통과버전": r["accepted_version"]
    })

pd.DataFrame(rows)

,번호,문장,재생성횟수,최종식,통과버전
0,1,모든 용기(vessel)의 설계압력은 운전압력의 1.1배 이상이어야 한다.,1,"forEvery(allOf(""Vessel""), isAtLeast(prop(x, ""D...",v2_pass
1,2,안전밸브가 있는 모든 장비는 설계압력이 정의되어 있어야 한다.,1,"forEvery(withSafetyValve(""Equipment""), exists(...",v2_pass
2,3,모든 열교환기의 설계온도는 운전온도보다 30 이상 높아야 한다.,1,"forEvery(allOf(""HeatExchanger""), isAtLeast(pro...",v2_pass
3,4,모든 장비 태그는 정해진 태그 패턴과 일치해야 한다.,1,"forEvery(allOf(""Equipment""), matchesPattern(pr...",v2_pass
4,5,모든 펌프는 재질과 설계압력이 모두 정의되어 있어야 한다.,1,"forEvery(allOf(""Pump""), and(exists(prop(x, ""Ma...",v2_pass
5,6,어떤 탱크도 운전압력이 2 barg를 초과해서는 안 된다.,1,"forEvery(allOf(""Tank""), isAtMost(prop(x, ""Oper...",v2_pass
6,7,운전온도가 200 이상인 모든 용기는 설계온도가 240 이상이어야 한다.,1,"forEvery(allOf(""Vessel""), or(not(isAtLeast(pro...",v2_pass
7,8,안전밸브가 있는 모든 장비는 설계압력이 운전압력의 1.2배 이상이어야 한다.,1,"forEvery(withSafetyValve(""Equipment""), isAtLea...",v2_pass
8,9,"모든 압축기는 설계압력이 운전압력의 1.1배 이상이고, 동시에 설계온도가 운전온도보...",1,"forEvery(allOf(""Compressor""), and(isAtLeast(pr...",v2_pass
9,10,모든 열교환기의 재질은 SS304 또는 SS316 중 하나여야 한다.,1,"forEvery(allOf(""HeatExchanger""), or(isEqual(pr...",v2_pass


| 번호 | 최종 모듈 조합식 |
|---:|---|
| 1 | `forEvery(allOf("Vessel"), isAtLeast(prop(x, "DesignPressure"), times(prop(x, "OperatingPressure"), 1.1)))` |
| 2 | `forEvery(withSafetyValve("Equipment"), exists(prop(x, "DesignPressure")))` |
| 3 | `forEvery(allOf("HeatExchanger"), isAtLeast(prop(x, "DesignTemperature"), plus(prop(x, "OperatingTemperature"), 30)))` |
| 4 | `forEvery(allOf("Equipment"), matchesPattern(prop(x, "Tag"), "TAG_PATTERN"))` |
| 5 | `forEvery(allOf("Pump"), and(exists(prop(x, "Material")), exists(prop(x, "DesignPressure"))))` |
| 6 | `forEvery(allOf("Tank"), isAtMost(prop(x, "OperatingPressure"), 2))` |
| 7 | `forEvery(allOf("Vessel"), or(not(isAtLeast(prop(x, "OperatingTemperature"), 200)), isAtLeast(prop(x, "DesignTemperature"), 240)))` |
| 8 | `forEvery(withSafetyValve("Equipment"), isAtLeast(prop(x, "DesignPressure"), times(prop(x, "OperatingPressure"), 1.2)))` |
| 9 | `forEvery(allOf("Compressor"), and(isAtLeast(prop(x, "DesignPressure"), times(prop(x, "OperatingPressure"), 1.1)), isAtLeast(prop(x, "DesignTemperature"), plus(prop(x, "OperatingTemperature"), 30))))` |
| 10 | `forEvery(allOf("HeatExchanger"), or(isEqual(prop(x, "Material"), "SS304"), isEqual(prop(x, "Material"), "SS316")))` |
| 11 | `forEvery(allOf("Equipment"), isAtLeast(prop(x, "DesignTemperature"), prop(x, "OperatingTemperature")))` |
| 12 | `forEvery(allOf("Pump"), and(matchesPattern(prop(x, "Tag"), "TAG_PATTERN"), exists(prop(x, "DesignPressure"))))` |

##### 4-4. 실패 로그 표

In [5]:
fail_rows=[]

for r in final_rules:
    for f in r["failures"]:
        fail_rows.append({
            "번호": r["id"],
            "실패버전": f["version"],
            "실패식": f["expr"],
            "실패사유": " | ".join(f["errors"])
        })

pd.DataFrame(fail_rows)

,번호,실패버전,실패식,실패사유
0,1,v1_fail,"forEvery(all(""Vessel""), isAtLeast(prop(x, ""Des...",root.forEvery.target: selector 위치에는 allOf/with...
1,2,v1_fail,"forEvery(withSafetyValve(""Equipment""), defined...",root.forEvery.condition: 명세에 없는 함수 'defined'가 ...
2,3,v1_fail,"forEvery(allOf(""HeatExchanger""), greaterThan(p...",root.forEvery.condition: 명세에 없는 함수 'greaterTha...
3,4,v1_fail,"forEvery(allOf(""Equipment""), matches(prop(x, ""...",root.forEvery.condition: 명세에 없는 함수 'matches'가 ...
4,5,v1_fail,"forEvery(allOf(""Pump""), and(exists(prop(x, ""Ma...",root.forEvery.condition: and는 최소 2개의 조건 인자가 필요...
5,6,v1_fail,"not(exists(isGreater(prop(x, ""OperatingPressur...",최상위 함수는 forEvery여야 합니다. 현재 'not' | forEvery는 t...
6,7,v1_fail,"forEvery(where(allOf(""Vessel""), isAtLeast(prop...",root.forEvery.target: selector 위치에는 allOf/with...
7,8,v1_fail,"forEvery(withSafetyValve(""Equipment""), isAtLea...",root.forEvery.condition.isAtLeast[1]: value 위치...
8,9,v1_fail,"forEvery(allOf(""Compressor""), isAtLeast(prop(x...","forEvery는 target, condition 인자 2개가 필요합니다. 현재 3개"
9,10,v1_fail,"forEvery(allOf(""HeatExchanger""), in(prop(x, ""M...",root.forEvery.condition: 명세에 없는 함수 'in'가 사용되었습니다.


### 5. 평가 지표

| 지표 | 값 | 의미 |
|---|---:|---|
| 최종 변환 성공률 | 12/12 = 100% | 12개 문장 모두 최종 AST가 게이트 통과 |
| 평균 재생성 횟수 | 1.0회 | 문장당 실패 후보 1개 후 수정본 통과 |
| 명세 외 함수 차단 | 8건 이상 | `all`, `defined`, `greaterThan`, `matches`, `where`, `multiply`, `in`, `isLess` 차단 |
| 구조 오류 차단 | 2건 | 최상위 `forEvery` 누락, `forEvery` 인자 개수 오류 |
| 이름 검증 차단 | 1건 | `TAG_REGEX` 차단 |

### 6. 한계와 규모가 커질 때 깨지는 지점

- 현재 validator는 문법/구조 검증 중심이다. 식이 문법적으로 맞아도 자연어 의미와 정확히 일치하는지는 별도 의미 검증이 필요하다.
- `Vessel`의 범위, 단위 변환, 압력 단위 barg/MPa, 온도 단위 ℃/K 같은 도메인 온톨로지 정보는 현재 코드에 하드코딩되어 있다.
- 조건부 규칙을 `or(not(A), B)`로 우회했지만, 실제 시스템에서는 `where`, `filter`, `implies` 같은 명시적 모듈을 추가하는 편이 사람이 읽기 쉽다.
- class/property 이름이 늘어나면 허용 목록을 코드에 직접 넣기보다 온톨로지 스키마 파일에서 로딩해야 한다.
- LLM 재생성을 실제 API로 연결할 경우, 실패 사유를 prompt에 넣되, 최종 수용 여부는 반드시 validator가 결정해야 한다.

### 7. 내가 떠올린 질문

1. `vessel`은 압력용기만 의미하는가, 아니면 tank도 포함하는 상위 개념인가?
2. `Equipment`는 Pump, Tank, HeatExchanger, Compressor를 모두 포함하는 최상위 클래스인가?
3. 설계압력/운전압력의 단위가 항상 barg라고 가정해도 되는가?
4. 설계온도/운전온도는 섭씨 기준인가?
5. `정의되어 있다`는 공란이 아니면 충분한가, 아니면 0이나 `N/A`도 미정의로 보아야 하는가?
6. 태그 패턴 `TAG_PATTERN`의 실제 정규식은 어디에서 관리되는가?
7. `withSafetyValve("Equipment")`는 안전밸브가 직접 연결된 장비만 의미하는가, 아니면 보호 대상 장비까지 포함하는가?
8. 재질 값은 대소문자와 공백을 정규화한 뒤 비교해야 하는가?
9. `SS304`, `SS316` 외에 `SUS304`, `304SS` 같은 동의어도 허용해야 하는가?
10. 실제 실행 엔진에서는 조건을 만족하지 않는 인스턴스 목록과 위반 사유를 어떤 포맷으로 반환해야 하는가?
